# Sleep Debt Data Cleaning
Cleans `bedtime_screentime_sleep_debt.csv` and saves a file that is ready for the **staging** layer.

Steps: load → inspect → column names → text → data types → missing values → duplicates → checks → save

## 1. Load the data

In [1]:
import pandas as pd

raw_path = "../data/bedtime_screentime_sleep_debt.csv"
clean_path = "../data/bedtime_screentime_sleep_debt_clean.csv"

df = pd.read_csv(raw_path)
rows_raw = len(df)
print("Rows, columns:", df.shape)
df.head()

Rows, columns: (8500, 18)


,user_id,age,gender,occupation_type,chronotype,bedtime_phone_minutes,primary_bedtime_app,screen_brightness_pct,blue_light_filter_active,caffeine_post_5pm_mg,physical_activity_min,sleep_latency_min,total_sleep_hours,deep_sleep_pct,rem_sleep_pct,morning_alarm_snoozes,next_day_fatigue_score,sleep_debt_category
0,USR-00001,29,Female,Healthcare / Shift Worker,Intermediate,179,Instagram / Reddit,54,1,0,21,84.4,3.20,23.5,22.2,7,10.0,Severe Sleep Debt
1,USR-00002,58,Female,Student,Intermediate,163,YouTube,76,1,92,72,86.7,4.06,18.0,21.5,7,10.0,Severe Sleep Debt
2,USR-00003,41,Female,Healthcare / Shift Worker,Night Owl,100,TikTok / Reels,59,0,45,23,70.5,3.20,18.1,20.5,7,10.0,Severe Sleep Debt
3,USR-00004,36,Non-Binary,Remote Tech,Intermediate,34,YouTube,52,0,97,49,38.1,6.99,21.6,22.4,1,1.7,Mild Deficit
4,USR-00005,23,Female,Healthcare / Shift Worker,Intermediate,27,TikTok / Reels,82,1,0,38,32.3,4.58,24.1,22.8,4,5.3,Moderate Debt


## 2. Inspect the data

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8500 entries, 0 to 8499
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8500 non-null   str    
 1   age                       8500 non-null   int64  
 2   gender                    8500 non-null   str    
 3   occupation_type           8500 non-null   str    
 4   chronotype                8500 non-null   str    
 5   bedtime_phone_minutes     8500 non-null   int64  
 6   primary_bedtime_app       8500 non-null   str    
 7   screen_brightness_pct     8500 non-null   int64  
 8   blue_light_filter_active  8500 non-null   int64  
 9   caffeine_post_5pm_mg      8500 non-null   int64  
 10  physical_activity_min     8500 non-null   int64  
 11  sleep_latency_min         8500 non-null   float64
 12  total_sleep_hours         8500 non-null   float64
 13  deep_sleep_pct            8500 non-null   float64
 14  rem_sleep_pct      

In [3]:
# Missing values, duplicates and number summary
print("Missing values per column:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
df.describe().T

Missing values per column:
user_id                     0
age                         0
gender                      0
occupation_type             0
chronotype                  0
bedtime_phone_minutes       0
primary_bedtime_app         0
screen_brightness_pct       0
blue_light_filter_active    0
caffeine_post_5pm_mg        0
physical_activity_min       0
sleep_latency_min           0
total_sleep_hours           0
deep_sleep_pct              0
rem_sleep_pct               0
morning_alarm_snoozes       0
next_day_fatigue_score      0
sleep_debt_category         0
dtype: int64

Duplicate rows: 0


,count,mean,std,min,25%,50%,75%,max
age,8500.0,34.355647,11.584259,18.0,25.00,32.00,42.00,65.0
bedtime_phone_minutes,8500.0,59.250000,36.931991,1.0,31.00,51.00,79.00,180.0
screen_brightness_pct,8500.0,55.153882,19.850550,10.0,42.00,55.00,69.00,100.0
blue_light_filter_active,8500.0,0.467765,0.498989,0.0,0.00,0.00,1.00,1.0
caffeine_post_5pm_mg,8500.0,33.222471,51.935645,0.0,0.00,0.00,55.00,250.0
physical_activity_min,8500.0,35.701412,20.819390,0.0,21.00,35.00,50.00,112.0
sleep_latency_min,8500.0,40.671471,17.444914,6.0,28.20,37.60,49.70,123.3
total_sleep_hours,8500.0,6.266209,1.446258,3.2,5.24,6.33,7.35,9.8
deep_sleep_pct,8500.0,21.735741,3.069377,8.1,19.70,21.90,23.90,28.0
rem_sleep_pct,8500.0,19.105859,2.546459,9.6,17.40,19.10,20.80,27.0


## 3. Standardize column names (lowercase snake_case)

In [4]:
df.columns = (
    df.columns.str.strip()
              .str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True)
              .str.strip("_")
)
list(df.columns)

['user_id',
 'age',
 'gender',
 'occupation_type',
 'chronotype',
 'bedtime_phone_minutes',
 'primary_bedtime_app',
 'screen_brightness_pct',
 'blue_light_filter_active',
 'caffeine_post_5pm_mg',
 'physical_activity_min',
 'sleep_latency_min',
 'total_sleep_hours',
 'deep_sleep_pct',
 'rem_sleep_pct',
 'morning_alarm_snoozes',
 'next_day_fatigue_score',
 'sleep_debt_category']

## 4. Clean text columns (remove extra spaces, fix casing)

In [5]:
text_cols = ["user_id", "gender", "occupation_type", "chronotype",
             "primary_bedtime_app", "sleep_debt_category"]

for col in text_cols:
    df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

df["user_id"] = df["user_id"].str.upper()

# Same value written in different cases (e.g. 'female' vs 'Female') -> one spelling
for col in ["gender", "occupation_type", "chronotype",
            "primary_bedtime_app", "sleep_debt_category"]:
    most_common = df.groupby(df[col].str.lower())[col].agg(lambda s: s.value_counts().idxmax())
    df[col] = df[col].str.lower().map(most_common)

df[text_cols].nunique()

user_id                8500
gender                    3
occupation_type           5
chronotype                3
primary_bedtime_app       6
sleep_debt_category       4
dtype: int64

## 5. Fix data types

In [6]:
int_cols = ["age", "bedtime_phone_minutes", "screen_brightness_pct",
            "caffeine_post_5pm_mg", "physical_activity_min", "morning_alarm_snoozes"]
float_cols = ["sleep_latency_min", "total_sleep_hours", "deep_sleep_pct",
              "rem_sleep_pct", "next_day_fatigue_score"]

# Numbers: anything that is not a number becomes missing (NaN)
for col in int_cols + float_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Boolean 0/1 column
df["blue_light_filter_active"] = (
    df["blue_light_filter_active"].astype(str).str.strip().str.lower()
      .map({"1": 1, "true": 1, "yes": 1, "0": 0, "false": 0, "no": 0})
)
df.dtypes

user_id                         str
age                           int64
gender                          str
occupation_type                 str
chronotype                      str
bedtime_phone_minutes         int64
primary_bedtime_app             str
screen_brightness_pct         int64
blue_light_filter_active      int64
caffeine_post_5pm_mg          int64
physical_activity_min         int64
sleep_latency_min           float64
total_sleep_hours           float64
deep_sleep_pct              float64
rem_sleep_pct               float64
morning_alarm_snoozes         int64
next_day_fatigue_score      float64
sleep_debt_category             str
dtype: object

## 6. Handle missing values

In [7]:
# Rows without a user_id cannot be used
df = df.dropna(subset=["user_id"])

# Numbers -> median, text -> 'Unknown'
for col in int_cols + float_cols + ["blue_light_filter_active"]:
    df[col] = df[col].fillna(df[col].median())
for col in text_cols:
    df[col] = df[col].fillna("Unknown")

df.isna().sum().sum()

np.int64(0)

## 7. Remove duplicates

In [8]:
before = len(df)
df = df.drop_duplicates()                       # identical rows
df = df.drop_duplicates(subset="user_id")       # same user twice -> keep first
duplicates_removed = before - len(df)
print("Duplicates removed:", duplicates_removed)

Duplicates removed: 0


## 8. Check for impossible values

In [9]:
valid = (
    df["age"].between(16, 100)
    & df["bedtime_phone_minutes"].between(0, 1440)
    & df["screen_brightness_pct"].between(0, 100)
    & (df["caffeine_post_5pm_mg"] >= 0)
    & (df["physical_activity_min"] >= 0)
    & (df["sleep_latency_min"] >= 0)
    & df["total_sleep_hours"].between(0, 24)
    & df["deep_sleep_pct"].between(0, 100)
    & df["rem_sleep_pct"].between(0, 100)
    & ((df["deep_sleep_pct"] + df["rem_sleep_pct"]) <= 100)
    & (df["morning_alarm_snoozes"] >= 0)
    & df["next_day_fatigue_score"].between(1, 10)
)
invalid_removed = int((~valid).sum())
df = df[valid]
print("Rows with impossible values removed:", invalid_removed)

Rows with impossible values removed: 0


## 9. Final types and save

In [10]:
# Whole numbers back to integers
for col in int_cols + ["blue_light_filter_active"]:
    df[col] = df[col].astype(int)

df = df.sort_values("user_id").reset_index(drop=True)
df.to_csv(clean_path, index=False, lineterminator="\n")   # '\n' works with BULK INSERT
print("Saved:", clean_path)
df.head()

Saved: ../data/bedtime_screentime_sleep_debt_clean.csv


,user_id,age,gender,occupation_type,chronotype,bedtime_phone_minutes,primary_bedtime_app,screen_brightness_pct,blue_light_filter_active,caffeine_post_5pm_mg,physical_activity_min,sleep_latency_min,total_sleep_hours,deep_sleep_pct,rem_sleep_pct,morning_alarm_snoozes,next_day_fatigue_score,sleep_debt_category
0,USR-00001,29,Female,Healthcare / Shift Worker,Intermediate,179,Instagram / Reddit,54,1,0,21,84.4,3.20,23.5,22.2,7,10.0,Severe Sleep Debt
1,USR-00002,58,Female,Student,Intermediate,163,YouTube,76,1,92,72,86.7,4.06,18.0,21.5,7,10.0,Severe Sleep Debt
2,USR-00003,41,Female,Healthcare / Shift Worker,Night Owl,100,TikTok / Reels,59,0,45,23,70.5,3.20,18.1,20.5,7,10.0,Severe Sleep Debt
3,USR-00004,36,Non-Binary,Remote Tech,Intermediate,34,YouTube,52,0,97,49,38.1,6.99,21.6,22.4,1,1.7,Mild Deficit
4,USR-00005,23,Female,Healthcare / Shift Worker,Intermediate,27,TikTok / Reels,82,1,0,38,32.3,4.58,24.1,22.8,4,5.3,Moderate Debt


## 10. Data quality summary

In [11]:
quality = pd.DataFrame({
    "metric": ["Rows in (raw)", "Rows out (clean)", "Rows removed",
               "Duplicates removed", "Impossible values removed", "Missing values left"],
    "value": [rows_raw, len(df), rows_raw - len(df),
              duplicates_removed, invalid_removed, int(df.isna().sum().sum())],
})
quality.to_csv("../data/bedtime_screentime_sleep_debt_quality_log.csv", index=False)
quality

,metric,value
0,Rows in (raw),8500
1,Rows out (clean),8500
2,Rows removed,0
3,Duplicates removed,0
4,Impossible values removed,0
5,Missing values left,0
